SACの参考資料

https://www.dskomei.com/entry/2022/06/30/222228

In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128


True


In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
12.8
NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [3]:
#!pip install torch

In [4]:
#!pip install cvxpy

In [5]:
import ecos
print("ECOS OK")

ECOS OK


In [6]:
!pip install ecos


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from pathlib import Path #Path クラスは、ファイルシステムへのパスを扱うための便利な方法を提供します。
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm  #進捗バーを簡単に追加するための便利なライブラリです。tqdm を使うと、ループ処理や時間のかかる計算タスクの進捗を視覚化することができます。
import seaborn as sns  #データ可視化を行うためのライブラリです。Matplotlibの上に構築されてい
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.distributions import Normal
import gymnasium as gym
import math

In [8]:
import torch

def model_dynamics(x: torch.Tensor, a: torch.Tensor, dt: float = 0.05) -> torch.Tensor:
    """
    Pendulum-v1 の離散時間 Dynamics を分析的に実装。
      x = [cosθ, sinθ, θ̇]
      u = a.squeeze(-1)
      θ̈ = −3g/(2l) sinθ + 3/(m l^2) u
      θ_{t+1} = θ + θ̇·dt
      θ̇_{t+1} = θ̇ + θ̈·dt
    """
    g, l, m = 10.0, 1.0, 1.0
    cos_th, sin_th, thdot = x[:,0], x[:,1], x[:,2]
    th   = torch.atan2(sin_th, cos_th)
    u    = a.squeeze(-1)
    thdd = -3 * g/(2*l) * torch.sin(th) + 3.0/(m*l**2) * u

    th_next    = th + thdot * dt
    thdot_next = thdot + thdd  * dt

    return torch.stack([torch.cos(th_next),
                        torch.sin(th_next),
                        thdot_next],
                       dim=1)


In [9]:
import cvxpy as cp

def solve_direction_full(W1, W2):
    W1_np = W1.detach().cpu().numpy()
    W2_np = W2.detach().cpu().numpy()
    d = len(W1_np)

    # e は 67,000 次元の最適化変数
    e = cp.Variable(d)

    # 目的関数：e · W2 を最大化
    objective = cp.Maximize(e @ W2_np)

    # 制約1：e · W1 >= 0（安全性を悪化させない）
    # 制約2：||e|| <= ||W2||（更新量の制限）
    constraints = [
        e @ W1_np >= 0,
        cp.norm(e, 2) <= np.linalg.norm(W2_np)
    ]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS)

    return torch.tensor(e.value, dtype=W1.dtype, device=W1.device)


In [10]:
import cvxpy as cp
import numpy as np

def solve_direction_cvxpy(W1, W2):
    W1_np = W1.detach().cpu().numpy()
    W2_np = W2.detach().cpu().numpy()

    """print("W1 norm =", np.linalg.norm(W1_np))
    print("W2 norm =", np.linalg.norm(W2_np))

    print("W1 nan =", np.isnan(W1_np).any())
    print("W2 nan =", np.isnan(W2_np).any())

    print("W1 inf =", np.isinf(W1_np).any())
    print("W2 inf =", np.isinf(W2_np).any())

    print("max abs W1 =", np.max(np.abs(W1_np)))
    print("max abs W2 =", np.max(np.abs(W2_np)))"""
    #d  = len(W1)

    #e = cp.Variable(d)
    a = cp.Variable()
    b = cp.Variable()

    e = a * W2_np + b * W1_np

    objective = cp.Maximize(e @ W2_np)
    constraints = [
        e @ W1_np >= 0,
        cp.norm(e, 2) <= np.linalg.norm(W2_np)
    ]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS,verbose=False)  # ECOS
    #prob.solve(solver=cp.ECOS, verbose=True)

    #print(prob.status)
    #print(a.value)
    #print(b.value)

    a_val = float(a.value)
    b_val = float(b.value)

    #print(np.linalg.norm(W1_np))
    #print(np.linalg.norm(W2_np))
    

    return (a_val * W2 + b_val * W1)#torch.tensor(e.value, dtype=torch.float32)


In [11]:
gym_name = 'Pendulum-v1'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 123456
torch.manual_seed(seed)
np.random.seed(seed)


In [12]:
#import torch
#import torch.nn as nn
#import torch.nn.functional as F
#from torch.distributions import Normal


class ClippedCriticNet(nn.Module):

    def __init__(self, input_num, output_num, hidden_size):

        super().__init__()

        self.linear1 = nn.Linear(input_num, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)
        self.linear3 = nn.Linear(hidden_size, output_num)

        self.linear4 = nn.Linear(input_num, hidden_size)
        self.linear5 = nn.Linear(hidden_size, hidden_size)
        self.linear6 = nn.Linear(hidden_size, output_num)

    def forward(self, state, action):
        xu = torch.cat([state, action], 1)

        x1 = F.relu(self.linear1(xu)) ##network1
        x1 = F.relu(self.linear2(x1))
        x1 = self.linear3(x1)

        x2 = F.relu(self.linear4(xu)) ##network2
        x2 = F.relu(self.linear5(x2))
        x2 = self.linear6(x2)

        return x1, x2  #network1, network2の出力

Actor では、ネットワークの出力の際にエントロピー項を追加しています。この値が方策の更新時と Soft Q 関数の更新時の損失値を求めるために使われます。エントロピー項には、出力値の対数確率に
−log(1−y2)+ε
 を足しています。これは、出力値が上下限に張り付かないようにしており、探索範囲の拡大に寄与しています。

In [13]:
 #LOG_SIG_MAX = 2
 #LOG_SIG_MIN = -20
 #epsilon = 1e-6


class SoftActorNet(nn.Module):

    def __init__(self, input_num, output_num, hidden_size, action_scale):

        super().__init__()

        self.linear1 = nn.Linear(input_num, hidden_size)
        self.linear2 = nn.Linear(hidden_size, hidden_size)

        self.mean_linear = nn.Linear(hidden_size, output_num)
        self.log_std_linear = nn.Linear(hidden_size, output_num)

        self.action_scale = torch.tensor(action_scale, dtype=torch.float32)
        self.action_bias = torch.tensor(0.)

    def forward(self, state, LOG_SIG_MAX = 2, LOG_SIG_MIN = -20):
        x = F.relu(self.linear1(state))
        x = F.relu(self.linear2(x))
        mean = self.mean_linear(x)
        log_std = self.log_std_linear(x)
        log_std = torch.clamp(log_std, min=LOG_SIG_MIN, max=LOG_SIG_MAX)  #PyTorchでテンソルの値を指定した範囲に制限するための関数です。
        return mean, log_std   #出力は平均と（対数にした）分散

    def sample(self, state,epsilon = 1e-6):
        self.epsilon=epsilon
        mean, log_std = self.forward(state)
        std = log_std.exp()
        normal = Normal(mean, std)  #平均mean、標準偏std2の正規分布を定義
        x_t = normal.rsample()      #rsample() と sample() の違いは、rsample() はサンプリングの際に勾配追跡が可能になる点です（requires_grad=True のテンソルに対して有効）
        y_t = torch.tanh(x_t)
        action = y_t * self.action_scale + self.action_bias
        log_prob = normal.log_prob(x_t)       #与えられた値がその分布に従う確率密度関数（PDF）の対数値を計算します。
        log_prob -= torch.log(self.action_scale * (1 - y_t.pow(2)) + self.epsilon)
            #y はネットワークの出力値で、通常 tanh 関数を使用して制限されています。
            #log(1 − y²) は、tanh の特性を利用し、出力値が端（±1付近）に近いほど大きなペナルティを与える形で計算されます。
            #ε は、小さい定数であり、ゼロ除算を回避するために加えられます。

        log_prob = log_prob.sum(1, keepdim=True)    #テンソルの指定された次元で要素を加算する際に使用されるメソッドです。
                                                    #特に、keepdim=True を設定すると、計算後のテンソルの形状が元の次元を保持します（
        mean = torch.tanh(mean) * self.action_scale + self.action_bias
        return action, log_prob, mean

    def to(self, device):
        self.action_scale = self.action_scale.to(device)
        self.action_bias = self.action_bias.to(device)
        return super().to(device)



これらのネットワークを使って SAC モデルを設計します。ポイントになるのは、「update_parameters 関数」内での Critic と Actor の損失値を求めるところです。両方ともエントロピー項を加味した計算になっています。そして、最後に
α
 を最適化しています。

In [14]:
class SoftActorCriticModel(object):

    def __init__(self, state_num, action_num, action_scale, args, device):

        self.gamma = args['gamma']
        self.tau = args['tau']
        self.alpha = args['alpha']
        self.device = device
        self.target_update_interval = args['target_update_interval']
        self.updates = 0

                # 履歴記録（エピソード毎）
        self.history_safe_grad_norms = []      # safety 勾配ノルム（エピソード単位の平均または最終値）
        self.history_stability_grad_norms = [] # stability 勾配ノルム
        self.history_safe_grad_norms_stage1 = []   # ステージ1 用（オプション）
        self.history_stability_grad_norms_stage2 = [] # ステージ2 用（オプション）

        # ステージ２用の安全項スケールを args に追加しておく
        self.lambda_safe = args.get('lambda_safe', 1)

        self.actor_net = SoftActorNet(
            input_num=state_num, output_num=action_num, hidden_size=args['hidden_size'], action_scale=action_scale).to(self.device)

        self.critic_net = ClippedCriticNet(input_num=state_num + action_num, output_num=1, hidden_size=args['hidden_size']).to(device=self.device)

        self.critic_net_target = ClippedCriticNet(input_num=state_num + action_num, output_num=1, hidden_size=args['hidden_size']).to(self.device)

        hard_update(self.critic_net_target, self.critic_net)
        convert_network_grad_to_false(self.critic_net_target)

        self.actor_optim = optim.Adam(self.actor_net.parameters(),lr=1e-4)
        self.critic_optim = optim.Adam(self.critic_net.parameters(),lr=args.get('critic_lr', 3e-4))

        self.critic_safe        = ClippedCriticNet(input_num=state_num + action_num, output_num=1, hidden_size=args['hidden_size']).to(device=self.device)
        self.critic_safe_target = ClippedCriticNet(input_num=state_num + action_num, output_num=1, hidden_size=args['hidden_size']).to(self.device)
        hard_update(self.critic_safe_target, self.critic_safe)
        convert_network_grad_to_false(self.critic_safe_target)
        self.critic_safe_optim  = optim.Adam(self.critic_safe.parameters(),lr=5e-5)


        self.target_entropy = -torch.prod(torch.Tensor(action_num).to(self.device)).item()  #target entropy H_tは、行動の次元数dに対して、  H_t=-d
                    #torch.prod関数を使用して、テンソル内の全要素の積を計算し、その結果をPythonの数値型（item()）に変換する
        self.log_alpha = torch.zeros(1, requires_grad=True, device=self.device)  #torch.zeros() を使用して 値がすべて0のテンソル を作成しています。1要素だけのテンソルを作成します。
        self.alpha_optim = optim.Adam([self.log_alpha])   #log_alphaのパラメータを最適化するためのAdamオプティマイザーを初期化します。

    def select_action(self, state, evaluate=False):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        if not evaluate:
            action, _, _ = self.actor_net.sample(state)
        else:
            _, _, action = self.actor_net.sample(state)
        return action.cpu().detach().numpy().reshape(-1)

    def h(self, x: torch.Tensor) -> torch.Tensor:
        return  x[:, 2] + 2.0

    def compute_safety_reward(self, s, a, s_next, alpha):
        """
        Control Barrier Function に基づく瞬時安全報酬 r_safe
        r_safe = exp(min(h(s') + (γ0-1)*h(s), 0))
        """
        h_s      = self.h(s)
        h_s_next = self.h(s_next)
        raw      = h_s_next + (alpha - 1.0) * h_s
        clipped = torch.clamp(raw, max=0.0)
        return torch.exp(clipped) - 1.0

    def compute_nav_reward(self, s, a, s_next):
        """
        既存の安定／目標達成報酬 r_nav を返すラッパー
        """
        return self.env_reward(s, a, s_next)

    def update_critics_and_actor(self, batch, episode_index):
        """
        Stage1/Stage2 の切り替えを含む
        - batch: replay buffer から取ってきたミニバッチ
        - episode_index: 現在のエピソード番号（1始まり）
        """
        self.updates += 1

        #max_grad_norm = 500.0   # <<< 追加###############

        stage1 = (episode_index <= args['stage1_episodes'])

        # 1) バッチの展開
        s, a, reward_tuple, s_next, mask = batch
        s      = s.to(self.device)
        a      = a.to(self.device)
        s_next = s_next.to(self.device)
        mask   = mask.to(self.device)
        # (必要に応じて mask, logp_old を batch に含めてください)
        reward_tensor = torch.FloatTensor(reward_tuple).to(self.device)
        r_nav, r_safe = reward_tensor[:, 0].unsqueeze(1), \
                                 reward_tensor[:, 1].unsqueeze(1)

        # 2) Critic 更新（常に行う）
        with torch.no_grad():
            a_next, logp_next, _ = self.actor_net.sample(s_next)
            q1_t, q2_t = self.critic_net_target(s_next, a_next)
            q_min     = torch.min(q1_t, q2_t) - self.alpha * logp_next
            target_nav  = r_nav  + mask.unsqueeze(1) * self.gamma * q_min
            # 安全性クリティックのターゲット
            qs1_t, qs2_t = self.critic_safe_target(s_next, a_next)
            qs_min       = torch.min(qs1_t, qs2_t)
            target_safe  = r_safe + mask.unsqueeze(1) * self.gamma * qs_min

        #q1, q2     = self.critic_net(s, a)
        #loss_nav   = F.mse_loss(q1, target_nav) + F.mse_loss(q2, target_nav)
        if not stage1:
            q1, q2   = self.critic_net(s, a)
            loss_nav = F.mse_loss(q1, target_nav) + F.mse_loss(q2, target_nav)
            q_mean   = torch.min(q1, q2).mean().item()
        else:
            loss_nav = None  # Stage1 は CriticNav 更新をスキップ
            q_mean   = None

        #q_mean   = torch.min(q1, q2).mean().item()
        #print(f"[DEBUG] loss_nav: {loss_nav.item():.3f}, Q1 mean: {q1.mean().item():.3f}")
        qs1, qs2   = self.critic_safe(s, a)
        loss_safe  = F.mse_loss(qs1, target_safe) + F.mse_loss(qs2, target_safe)

        # Critic
        if loss_nav is not None:
            self.critic_optim.zero_grad()

            loss_nav.backward()

            # [Compare] gradient clipping unified with Proposal_Compare (Critic: 500)
            max_grad_norm = 500.0
            torch.nn.utils.clip_grad_norm_(self.critic_net.parameters(), max_grad_norm)

            self.critic_optim.step()

        self.critic_safe_optim.zero_grad()

        loss_safe.backward()

        # [Compare] gradient clipping unified with Proposal_Compare (Safety Critic: 500)
        max_grad_norm = 500.0
        torch.nn.utils.clip_grad_norm_(self.critic_safe.parameters(), max_grad_norm)

        self.critic_safe_optim.step()

        # --- 安全クリティック勾配ノルムを計算して保持 ---
        total_sq = 0.0
        for p in self.critic_safe.parameters():
            if p.grad is not None:
                total_sq += float(p.grad.data.norm(2).item()) ** 2
        critic_safe_grad_norm = total_sq ** 0.5
        self.last_safe_grad_norm = critic_safe_grad_norm


        # 3) Actor 更新
        # stability_loss と safety_loss を定義
        pi, logp_pi, _ = self.actor_net.sample(s)
        q1_pi, q2_pi   = self.critic_net(s, pi)
        q_min_pi       = torch.min(q1_pi, q2_pi)
        stability_loss = (self.alpha * logp_pi - q_min_pi).mean()

        # safety_loss
        qs1_pi, qs2_pi    = self.critic_safe(s, pi)
        safety_loss_unscaled = - torch.min(qs1_pi, qs2_pi).mean()
        safety_loss = self.lambda_safe * safety_loss_unscaled
        #safety_loss_stage = - self.lambda_safe * torch.min(qs1_pi, qs2_pi).mean()



        # Stage判定
        #stage1 = (episode_index <= args['stage1_episodes'])

        if episode_index <= args['stage1_episodes']:
            # ステージ1：安全性のみで Actor 更新

            self.actor_optim.zero_grad()

            safety_loss.backward()
            self.actor_optim.step()

            nav_loss_item   = None
            safe_loss_item  = safety_loss.item()

        else:
            # ステージ2：restricted_direction を用いた制限付き更新
            # a) stability勾配
            self.actor_optim.zero_grad()

            stability_loss.backward(retain_graph=True)
            grad_st = torch.cat([p.grad.view(-1) for p in self.actor_net.parameters()])


            # b) safety勾配

            self.actor_optim.zero_grad()

            safety_loss.backward(retain_graph=True)
            grad_sa = torch.cat([p.grad.view(-1) for p in self.actor_net.parameters()])

            # 生のノルムを記録（正規化前の大きさ）
            st_norm_raw = grad_st.norm().item()
            sa_norm_raw = grad_sa.norm().item()
            self.history_stability_grad_norms.append(st_norm_raw)
            self.history_safe_grad_norms.append(sa_norm_raw)


            # d) 正規化 ＆ restricted_direction
            #eps = 1e-6
            #safe_scale = 0.2 #追加した
            #safety_loss2  = safety_loss_stage * safe_scaleこれはかんけいない
            #grad_st = grad_st / (grad_st.norm()  + eps)
            #grad_sa = grad_sa / (grad_sa.norm() + eps)
            #grad_sa = grad_sa * safe_scale

            #e = restricted_direction(grad_sa, grad_st)
            #print(len(e))

            # W1 = safety 勾配, W2 = stability 勾配
            W1 = grad_sa
            W2 = grad_st

            # CVXPY で最適化問題を解く
            #print("DEBUG W1 type:", type(W1), "W2 type:", type(W2))"""
            """eps = 1e-8
            grad_st = grad_st / (grad_st.norm() + eps)
            grad_sa = grad_sa / (grad_sa.norm() + eps)
            W1 = grad_sa 
            W2 = -grad_st"""

            

            e = solve_direction_cvxpy(W1, W2).to(self.device)


            """dot   = torch.dot(grad_st, grad_sa).item()
            norm1 = grad_st.norm().item()
            norm2 = grad_sa.norm().item()
            cos_sim = dot / (norm1 * norm2 + 1e-8)
            angle = math.degrees(math.acos(dot/(norm1*norm2+1e-8)))"""
            #print(f"[DEBUG] dot(W_nav,W_safe)={dot:.3f},cos={cos_sim:.3f}, angle={angle:.1f}°")


        # 展開前に p.grad を上書きするため、ここでは展開後に出力する

            # e) 各パラメータ勾配に展開
            idx = 0
            for p in self.actor_net.parameters():
                n = p.numel()
                p.grad = e[idx:idx+n].view_as(p).clone()
                idx += n

            # [Compare] gradient clipping added as a comparison-experiment common condition
            # (this clip does NOT exist in the original CAC algorithm; it is added purely to
            # match Proposal_Compare's Actor clip=50, and does not alter Eq.(10) / solve_direction_cvxpy)
            max_grad_norm = 50.0
            torch.nn.utils.clip_grad_norm_(self.actor_net.parameters(), max_grad_norm)

            # f) 更新
            self.actor_optim.step()
            """try:
                for g in self.actor_optim.param_groups:
                    g.setdefault('orig_lr', g['lr'])
                    g['lr'] = g['lr'] * 2.0

            # 勾配クリッピング（actor のパラメータ更新前）
                torch.nn.utils.clip_grad_norm_(self.actor_net.parameters(), max_norm=1.0)


            # actor の更新
                self.actor_optim.step()

            finally:
            # lr を必ず元に戻す
                for g in self.actor_optim.param_groups:
                    if 'orig_lr' in g:
                        g['lr'] = g['orig_lr']
                        del g['orig_lr']"""

            nav_loss_item  = stability_loss.item()
            safe_loss_item = safety_loss.item()

        # 4) Entropy α 更新（既存ロジック）
        alpha_loss = -(self.log_alpha * (logp_pi + self.target_entropy).detach()).mean()
        self.alpha_optim.zero_grad()
        alpha_loss.backward()
        self.alpha_optim.step()
        self.alpha = self.log_alpha.exp()

        # 5) ターゲットネットワークのソフト更新
        if self.updates % self.target_update_interval == 0:
            soft_update(self.critic_net_target, self.critic_net, self.tau)
            soft_update(self.critic_safe_target, self.critic_safe, self.tau)

                # === ノルム記録 ===
        # stage1 の場合は安全クリティックの勾配ノルム（self.last_safe_grad_norm が既に保持されている）
        # stage2 の場合は stability と safety 勾配の L2 ノルム（計算済み grad_st, grad_sa を使う）
        try:
            if stage1:
                # stage1: 安全クリティック勾配ノルム（CriticSafe の勾配ノルム）
                self.history_safe_grad_norms.append(self.last_safe_grad_norm if hasattr(self, 'last_safe_grad_norm') else 0.0)
                # stability はこの段階では未更新なので 0 を格納
                self.history_stability_grad_norms.append(0.0)
            else:
                # stage2: actor の stability/safety 勾配ノルム（grad_st, grad_sa は正規化前の値を使うのが望ましい）
                # grad_st, grad_sa はスコープ内で定義されているはずなので取得する
                st_norm = grad_st.norm().item() if 'grad_st' in locals() else 0.0
                sa_norm = grad_sa.norm().item() if 'grad_sa' in locals() else 0.0
                self.history_stability_grad_norms.append(st_norm)
                self.history_safe_grad_norms.append(sa_norm)
        except Exception:
            # 記録でエラーが出ても学習を止めない
            self.history_safe_grad_norms.append(0.0)
            self.history_stability_grad_norms.append(0.0)


        #return nav_loss_item, safe_loss_item, self.last_safe_grad_norm
                # 既存の最後の return をこの1行に置き換える
        # stage1 の場合 grad_st/grad_sa/dot は未定義の可能性があるため安全値を返す
        try:
            st_norm_raw = st_norm_raw if 'st_norm_raw' in locals() else 0.0
            sa_norm_raw = sa_norm_raw if 'sa_norm_raw' in locals() else self.last_safe_grad_norm if hasattr(self, 'last_safe_grad_norm') else 0.0
            dot_raw     = dot if 'dot' in locals() else 0.0
        except Exception:
            st_norm_raw, sa_norm_raw, dot_raw = 0.0, 0.0, 0.0

        return nav_loss_item, safe_loss_item, critic_safe_grad_norm, st_norm_raw, sa_norm_raw, dot_raw


In [15]:
import inspect
print(inspect.getsource(solve_direction_cvxpy))


def solve_direction_cvxpy(W1, W2):
    W1_np = W1.detach().cpu().numpy()
    W2_np = W2.detach().cpu().numpy()

    """print("W1 norm =", np.linalg.norm(W1_np))
    print("W2 norm =", np.linalg.norm(W2_np))

    print("W1 nan =", np.isnan(W1_np).any())
    print("W2 nan =", np.isnan(W2_np).any())

    print("W1 inf =", np.isinf(W1_np).any())
    print("W2 inf =", np.isinf(W2_np).any())

    print("max abs W1 =", np.max(np.abs(W1_np)))
    print("max abs W2 =", np.max(np.abs(W2_np)))"""
    #d  = len(W1)

    #e = cp.Variable(d)
    a = cp.Variable()
    b = cp.Variable()

    e = a * W2_np + b * W1_np

    objective = cp.Maximize(e @ W2_np)
    constraints = [
        e @ W1_np >= 0,
        cp.norm(e, 2) <= np.linalg.norm(W2_np)
    ]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS,verbose=False)  # ECOS
    #prob.solve(solver=cp.ECOS, verbose=True)

    #print(prob.status)
    #print(a.value)
    #print(b.value)

    a_val = float(a.value)
    b_val = floa

In [16]:
def soft_update(target, source, tau):
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(target_param.data * (1.0 - tau) + param.data * tau)


def hard_update(target, source):
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(param.data)


def convert_network_grad_to_false(network):
    for param in network.parameters():
        param.requires_grad = False

In [17]:
import random
import numpy as np


class ReplayMemory:

    def __init__(self, memory_size):
        self.memory_size = memory_size
        self.buffer = []
        self.position = 0

    def push(self, state, action, reward, next_state, mask):
        if len(self.buffer) < self.memory_size:
            self.buffer.append(None)
        self.buffer[self.position] = (state, action, reward, next_state, mask)
        self.position = (self.position + 1) % self.memory_size

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        state      = torch.from_numpy(state).float()
        action     = torch.from_numpy(action).float()
        reward     = torch.from_numpy(reward).float()
        next_state = torch.from_numpy(next_state).float()
        done       = torch.from_numpy(done).float()
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)

    def clear(self):
        # ここを追加
        self.buffer.clear()
        self.position = 0

In [18]:
import os
import time
import csv
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype']  = 42

args = {
    'gym_name' : 'Pendulum-v1',
    'gamma': 0.99,
    'tau': 0.005,
    'alpha': 0.2,
    'seed': 123456,
    'batch_size': 256,
    'hidden_size': 256,
    'start_steps': 1000,
    'updates_per_step': 1,
    'target_update_interval': 1,
    'memory_size': 100000,
    'epochs': 100,
    'eval_interval': 10,
    'stage1_episodes': 50,
    'stage2_episodes': 200,
    'log_interval':  10,
    # --- [Compare] comparison-experiment common conditions ---
    'cbf_alpha0': 0.2,     # CBF alpha0, fully separated from SAC entropy alpha
    'lambda_safe': 1,      # explicit (already the class default, kept for symmetry with Proposal_Compare)
}

BASE_SEED = 123456  # run_seed = BASE_SEED + run_id (per-run reproducible seeding, matches Proposal_Compare)
total_episodes = args['stage1_episodes'] + args['stage2_episodes']


def save_single_run(run_id, save_dir="runs_cac"):
    os.makedirs(save_dir, exist_ok=True)
    data = {
        "episode_rewards_nav": np.array(episode_rewards_nav),
        "episode_rewards_safe": np.array(episode_rewards_safe),
        "episode_violations": np.array(episode_violations),
        "episode_safe_grad_stage1": np.array(episode_safe_grad_stage1),
        "episode_safe_grad_stage2": np.array(episode_safe_grad_stage2),
        "episode_stability_grad": np.array(episode_stability_grad),
        "episode_grad_dot": np.array(episode_grad_dot),
        "episode_loss_safe": np.array(episode_loss_safe),
        "episode_loss_stability": np.array(episode_loss_stability),
        "episode_theta_dot_mean": np.array(episode_theta_dot_mean),
        "episode_theta_dot_max": np.array(episode_theta_dot_max),
        "episode_theta_dot_min": np.array(episode_theta_dot_min),
        "episode_eval_nav": np.array(episode_eval_nav),    # [Compare] evaluation reward, separate from training reward
        "episode_eval_safe": np.array(episode_eval_safe),  # [Compare]
        "runtime_sec": time.time() - start_time,
    }
    np.savez_compressed(f"{save_dir}/run_{run_id}.npz", **data)
    print(f"[SAVE] Saved run data to {save_dir}/run_{run_id}.npz")


def run_experiment(run_id):
    print(f"\n===== RUN {run_id} START =====")

    global episode_rewards_nav, episode_rewards_safe
    global episode_loss_min, episode_loss_max, episode_loss_safe, episode_loss_stability
    global episode_dot_neg_counts, episode_dot_neg_ratio
    global episode_violations
    global episode_safe_grad_stage1, episode_safe_grad_stage2
    global episode_stability_grad, episode_stability_grad_log10, episode_grad_dot, episode_grad_cos
    global episode_theta_dot_mean, episode_theta_dot_max, episode_theta_dot_min
    global episode_theta_dot_signed_min, episode_theta_dot_signed_max
    global episode_eval_nav, episode_eval_safe
    global n_steps, n_update, printed_stage1, start_time
    global env, agent, memory, total_stage1, buffer_fill

    episode_rewards_nav = []
    episode_rewards_safe = []
    episode_loss_min = []
    episode_loss_max = []
    episode_loss_safe = []
    episode_loss_stability = []
    episode_dot_neg_counts = []
    episode_dot_neg_ratio = []
    episode_violations = []
    episode_safe_grad_stage1 = []
    episode_safe_grad_stage2 = []
    episode_stability_grad = []
    episode_stability_grad_log10 = []
    episode_grad_dot = []
    episode_grad_cos = []
    episode_theta_dot_mean = []
    episode_theta_dot_max = []
    episode_theta_dot_min = []
    episode_theta_dot_signed_min = []
    episode_theta_dot_signed_max = []
    episode_eval_nav = []   # [Compare] deterministic evaluation reward (Nav), separate from training reward
    episode_eval_safe = []  # [Compare] deterministic evaluation reward (Safe), separate from training reward

    # [Compare] per-run reproducible seeding, identical scheme to Proposal_Compare
    run_seed = BASE_SEED + run_id
    random.seed(run_seed)
    np.random.seed(run_seed)
    torch.manual_seed(run_seed)

    env = gym.make(args['gym_name'])
    env.action_space.seed(run_seed)

    agent = SoftActorCriticModel(
        state_num=env.observation_space.shape[0],
        action_num=env.action_space.shape[0],
        action_scale=env.action_space.high[0],
        args=args,
        device=device
    )
    memory = ReplayMemory(args['memory_size'])

    n_steps = 0
    n_update = 0

    total_stage1 = args['stage1_episodes']
    buffer_fill  = args['batch_size']

    printed_stage1 = False
    start_time = time.time()

    for ep in range(1, total_episodes + 1):

        if ep == total_stage1 + 1:
            memory.clear()
            print(f"=== Entering Stage2 (ep={ep}), replay buffer cleared ===")

            # 2) データ再収集（NavCritic 用）
            state, _ = env.reset(seed=run_seed + ep)
            for _ in range(buffer_fill):
                if n_steps < args['start_steps']:
                    action = env.action_space.sample()
                else:
                    action = agent.select_action(state)

                next_state, r_env, terminated, truncated, _ = env.step(action)
                done = terminated or truncated

                # safety reward も計算してタプルにする
                s_t  = torch.from_numpy(state).float().to(device).unsqueeze(0)
                a_t  = torch.from_numpy(action).float().to(device).unsqueeze(0)
                ns_t = torch.from_numpy(next_state).float().to(device).unsqueeze(0)
                r_safe = agent.compute_safety_reward(s_t, a_t, ns_t, args['cbf_alpha0']).item()

                memory.push(
                    state=state,
                    action=action,
                    reward=(r_env, r_safe),
                    next_state=next_state,
                    mask=float(not done)
                )
                if not done:
                    state = next_state
                else:
                    state, _ = env.reset(seed=run_seed + ep)
                n_steps += 1
            # 2) NavCritic のプリトレーニング
            pretrain_batches = 1  # お好みで調整
            for _ in range(pretrain_batches):
                # バッチ取得
                s_b, a_b, r_b, s_next_b, mask_b = memory.sample(args['batch_size'])
                # デバイス転送
                s_b = s_b.to(device); a_b = a_b.to(device)
                s_next_b = s_next_b.to(device); mask_b = mask_b.to(device)

                # r_nav のみ取り出し
                r_nav = torch.FloatTensor(r_b)[:, 0].unsqueeze(1).to(device)

                # ターゲット Q 値計算
                with torch.no_grad():
                    a_next, logp_next, _ = agent.actor_net.sample(s_next_b)
                    q1_t, q2_t = agent.critic_net_target(s_next_b, a_next)
                    q_min = torch.min(q1_t, q2_t) - agent.alpha * logp_next
                    target_nav = r_nav + mask_b.unsqueeze(1) * agent.gamma * q_min

                # CriticNav 更新
                q1, q2 = agent.critic_net(s_b, a_b)
                loss_nav = F.mse_loss(q1, target_nav) + F.mse_loss(q2, target_nav)
                agent.critic_optim.zero_grad()
                loss_nav.backward()
                agent.critic_optim.step()
                soft_update(agent.critic_net_target, agent.critic_net, agent.tau)
            print(">>> CriticNav pre-training done. Stage2 Actor updates ready. <<<")


        ep_ret_safe = 0.0
        ep_ret_nav = 0.0

        # per-update temporary lists for this episode
        perupdate_st_norms = []
        perupdate_sa_norms = []
        perupdate_dots = []
        perupdate_safe_grad_norms = []

        loss_nav_list = []#ここの二行
        loss_safe_list = []

        violations    = 0
        done = False
        if ep == 1:
            state, _ = env.reset(seed=run_seed)
        else:
            state, _ = env.reset()

        # [ADDED START] per-episode theta_dot collection
        theta_dot_vals = []
        theta_dot_signed = []       # stores signed theta_dot for this episode
        # [ADDED END]

        while not done:

            if args['start_steps'] > n_steps:
                action = env.action_space.sample()
            else:
                action = agent.select_action(state)

            if len(memory) > args['batch_size']:
                for _ in range(args['updates_per_step']):
                    batch = memory.sample(args['batch_size'])
                    #loss_nav, loss_safe, safe_grad_norm, st_norm_raw, sa_norm_raw, dot_raw = agent.update_critics_and_actor(batch, episode_index=ep) #パラメータの更新 ここに変更加えた
                    #if loss_nav is not None:
                        #loss_nav_list.append(loss_nav)#ここも加えた
                    # call update and receive extended metrics
                    res = agent.update_critics_and_actor(batch, episode_index=ep)
                    if res is None:
                        loss_nav, loss_safe, safe_grad_norm, st_norm_raw, sa_norm_raw, dot_raw = (None, None, None, None, None, None)
                    else:
                        loss_nav, loss_safe, safe_grad_norm, st_norm_raw, sa_norm_raw, dot_raw = res

                    # original loss list append
                    if loss_safe is not None:
                        loss_safe_list.append(loss_safe)
                    if loss_nav is not None:
                        loss_nav_list.append(loss_nav)

                    # append per-update metrics into the episode-local buffers
                    if safe_grad_norm is not None:
                        perupdate_safe_grad_norms.append(float(safe_grad_norm))
                    if st_norm_raw is not None:
                        perupdate_st_norms.append(float(st_norm_raw))
                    if sa_norm_raw is not None:
                        perupdate_sa_norms.append(float(sa_norm_raw))
                    if dot_raw is not None:
                        perupdate_dots.append(float(dot_raw))
                    n_update += 1
            # エピソードごとの更新後、Stage1 最終エピソードならノルムを出力
        #if ep == args['stage1_episodes']:
           # print(f"=== Stage1 End: 安全クリティック勾配ノルム === "f"{safe_grad_norm:.6f}")


            next_state, r_env, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            state_t      = torch.from_numpy(state).float().to(device)
            next_state_t = torch.from_numpy(next_state).float().to(device)
            #ここに報酬計算
            #ナビゲーション報酬
            #ep_ret_nav = r_env#ここも加えた変更

            #安全報酬
            action_t = torch.from_numpy(action).float().to(device).unsqueeze(0)
            r_safe = agent.compute_safety_reward(
                 state_t.unsqueeze(0),
                 action_t,
                 next_state_t.unsqueeze(0),
                 args['cbf_alpha0']
             ).item()
             # --- Barrier 関数違反カウント ---
            # h(s_next)<0 を「違反」としてカウント
            if agent.h(next_state_t.unsqueeze(0)).item() < 0.0:
                violations += 1

             # 両者をタプルで保存
            memory.push(state=state, action=action,reward=(r_env, r_safe), next_state=next_state,mask = float(not done))

            # [ADDED START] collect theta_dot (Pendulum obs: [cos, sin, theta_dot])
            theta_dot = float(next_state[2])
            theta_dot_signed.append(theta_dot)           # record signed value
            theta_dot_vals.append(abs(theta_dot))  # store absolute angular velocity
            # [ADDED END]


            #else:
            #  done=False
            n_steps += 1
            ep_ret_nav += r_env
            ep_ret_safe+= r_safe

            state = next_state
        if ep == args['stage1_episodes'] and not printed_stage1:
            # agent.last_safe_grad_norm に最後計算されたノルムが入っている
            print(
                f"=== Stage1 End: 安全クリティック勾配ノルム === "
                f"{agent.last_safe_grad_norm:.6f}"
            )
            printed_stage1 = True
            torch.save(agent.actor_net.state_dict(), "actor_safe_only.pth")
            print(">>> 安全制御のみの係数を保存しました (actor_safe_only.pth) <<<")
            # --- エピソード単位での勾配メトリクス集計（per-update バッファの平均を保存） --
        # エピソード終了時：CriticSafe ノルムを共通の指標として計算
        if len(perupdate_safe_grad_norms) > 0:
            mean_safe = float(np.mean(perupdate_safe_grad_norms))
        else:
            mean_safe = float(getattr(agent, 'last_safe_grad_norm', 0.0))

        if ep <= args['stage1_episodes']:
            episode_safe_grad_stage1.append(mean_safe)
        else:
            # Stage2 でも CriticSafe のノルム（mean_safe）を使う
            episode_safe_grad_stage2.append(mean_safe)

            # 他の Stage2 指標は従来どおりエピソード平均を取る
            sa_mean = float(np.mean(perupdate_sa_norms)) if len(perupdate_sa_norms) > 0 else 0.0
            st_mean = float(np.mean(perupdate_st_norms)) if len(perupdate_st_norms) > 0 else 0.0
            dot_mean = float(np.mean(perupdate_dots)) if len(perupdate_dots) > 0 else 0.0

            episode_stability_grad.append(st_mean)
            episode_grad_dot.append(dot_mean)

            if st_mean > 0 and np.isfinite(st_mean):########
                episode_stability_grad_log10.append(np.log10(st_mean))
            else:
                episode_stability_grad_log10.append(np.nan)

            # W1・W2 < 0 の回数をカウント
            neg_count = sum(1 for d in perupdate_dots if d < 0)########
            episode_dot_neg_counts.append(neg_count)#######

                # W1・W2 < 0 の割合を記録
            if len(perupdate_dots) > 0:
                neg_ratio = sum(1 for d in perupdate_dots if d < 0) / len(perupdate_dots)
            else:
                neg_ratio = 0.0
            episode_dot_neg_ratio.append(neg_ratio)



        """if ep <= args['stage1_episodes']:
            if len(perupdate_safe_grad_norms) > 0:
                episode_safe_grad_stage1.append(float(np.mean(perupdate_safe_grad_norms)))
            else:
                episode_safe_grad_stage1.append(float(getattr(agent, 'last_safe_grad_norm', 0.0)))
        else:
            sa_mean = float(np.mean(perupdate_sa_norms)) if len(perupdate_sa_norms) > 0 else 0.0
            st_mean = float(np.mean(perupdate_st_norms)) if len(perupdate_st_norms) > 0 else 0.0
            dot_mean = float(np.mean(perupdate_dots)) if len(perupdate_dots) > 0 else 0.0

            episode_safe_grad_stage2.append(sa_mean)
            episode_stability_grad.append(st_mean)
            episode_grad_dot.append(dot_mean)"""


        # --- エピソード終了後に集計・リストへ追加 ---
        valid_nav = [ln for ln in loss_nav_list if ln is not None]
        if valid_nav:
            episode_loss_min.append(min(loss_nav_list))
            episode_loss_max.append(max(loss_nav_list))
        else:
            episode_loss_min.append(0.0)
            episode_loss_max.append(0.0)

        # SafeLoss と StabilityLoss を保存
        if len(loss_safe_list) > 0:
            episode_loss_safe.append(np.mean(loss_safe_list))
        else:
            episode_loss_safe.append(0.0)

        if len(loss_nav_list) > 0:
            episode_loss_stability.append(np.mean(loss_nav_list))
        else:
            episode_loss_stability.append(0.0)



        episode_violations.append(violations)
        episode_rewards_nav.append(ep_ret_nav)
        episode_rewards_safe.append(ep_ret_safe)

        # [ADDED START] Stage2: record theta_dot stats per episode
        if ep > args['stage1_episodes']:
            if len(theta_dot_vals) > 0:
                mean_td = float(np.mean(theta_dot_vals))
                max_td  = float(np.max(theta_dot_vals))
                min_td  = float(np.min(theta_dot_vals))              # [ADDED] abs 最小
            else:
                mean_td = 0.0
                max_td  = 0.0
                min_td = 0.0

            if len(theta_dot_signed) > 0:
                signed_min = float(np.min(theta_dot_signed))         # [ADDED] 符号付き最小（負の大きさ）
                signed_max = float(np.max(theta_dot_signed))         # [ADDED] 符号付き最大（正の大きさ）
            else:
                signed_min = 0.0; signed_max = 0.0

            episode_theta_dot_mean.append(mean_td)
            episode_theta_dot_max.append(max_td)
            episode_theta_dot_min.append(min_td)                    # [ADDED]
            episode_theta_dot_signed_min.append(signed_min)         # [ADDED]
            episode_theta_dot_signed_max.append(signed_max)         # [ADDED]

            # print every 10 Stage2 episodes (stage2_index counts from 1)
            stage2_index = ep - args['stage1_episodes']
            if stage2_index % 10 == 0:
                start_idx = max(0, len(episode_theta_dot_mean) - 10)
                recent_mean = episode_theta_dot_mean[start_idx:]
                recent_max  = episode_theta_dot_max[start_idx:]
                recent_min  = episode_theta_dot_min[start_idx:]      # [ADDED]
                recent_smin = episode_theta_dot_signed_min[start_idx:]  # [ADDED]
                recent_smax = episode_theta_dot_signed_max[start_idx:]  # [ADDED]
                print(
                f"[Stage2 Ep {stage2_index}] "
                f"θ̇ mean(abs) (last {len(recent_mean)} eps): {np.mean(recent_mean):.4f}, "
                f"θ̇ max(abs): {np.max(recent_max):.4f}, "
                f"θ̇ min(abs): {np.min(recent_min):.4f}, "
                f"θ̇ signed_min: {np.min(recent_smin):.4f}, "
                f"θ̇ signed_max: {np.max(recent_smax):.4f}"
            )
        # [ADDED END]

        # ログ出力（任意のタイミングで）
        if ep % args['log_interval'] == 0:
            print(f"[Ep {ep:03d}] "
                  f"NavR: {ep_ret_nav:.2f}, SafeR: {ep_ret_safe:.2f}, "
                  f"LossNav(min,max): ({episode_loss_min[-1]:.2f},{episode_loss_max[-1]:.2f}), "
                  f"Violations: {violations}")


        # [Compare] evaluation reward recorded into the run-level episode_eval_nav/safe lists
        # (declared once per run in run_experiment), instead of being reset every episode.
        if ep % args['eval_interval'] == 0:
            avg_nav = 0.0
            avg_safe =0.0
            #eval_navs = []#####
            #eval_safes =[]#####
            for _  in range(args['eval_interval']):
                state, _ = env.reset()
                ep_nav_eval = 0.0
                ep_safe_eval =0.0
                done = False
                while not done:
                    with torch.no_grad():
                        action = agent.select_action(state, evaluate=True)
                    next_state, r_nav_eval, terminated, truncated, _ = env.step(action)
                    done = terminated or truncated
                    # 外部報酬（ナビゲーション）
                    ep_nav_eval += r_nav_eval
                    # 安全報酬（Barrier 関数）
                    s_t  = torch.from_numpy(state).float().to(device).unsqueeze(0)
                    a_t  = torch.from_numpy(action).float().to(device).unsqueeze(0)
                    ns_t = torch.from_numpy(next_state).float().to(device).unsqueeze(0)
                    r_safe_eval = agent.compute_safety_reward(s_t, a_t, ns_t, args['cbf_alpha0']).item()
                    ep_safe_eval += r_safe_eval
                    state = next_state
                avg_nav  += ep_nav_eval
                avg_safe += ep_safe_eval
            avg_nav  /= args['eval_interval']
            avg_safe /= args['eval_interval']
            episode_eval_nav.append(avg_nav)
            episode_eval_safe.append(avg_safe)

            print("Episode: {}, Eval Avg Nav. Reward: {:.2f}, Eval Avg Safe. Reward: {:.2f}".format(ep, avg_nav, avg_safe))
            if ep == total_episodes:
                torch.save(agent.actor_net.state_dict(), "actor_safe_stable.pth")
                print(">>> 安定化制御込みの係数を保存しました (actor_safe_stable.pth) <<<")

    print('Game Done !! Max Eval Nav: {:.2f}, Max Eval Safe: {:.2f}'.format(
        np.max(episode_eval_nav) if episode_eval_nav else 0.0,
        np.max(episode_eval_safe) if episode_eval_safe else 0.0))

    save_single_run(run_id)
    print(f"===== RUN {run_id} END =====\n")


In [19]:
NUM_RUNS = 1
for run_id in range(1, NUM_RUNS + 1):
    run_experiment(run_id)



===== RUN 1 START =====


[Ep 010] NavR: -1265.78, SafeR: -15.73, LossNav(min,max): (0.00,0.00), Violations: 52


Episode: 10, Eval Avg Nav. Reward: -1442.20, Eval Avg Safe. Reward: -7.25


[Ep 020] NavR: -1666.65, SafeR: -2.66, LossNav(min,max): (0.00,0.00), Violations: 10


Episode: 20, Eval Avg Nav. Reward: -1489.17, Eval Avg Safe. Reward: -3.02


...

Episode: 230, Eval Avg Nav. Reward: -701.24, Eval Avg Safe. Reward: -15.32


[Stage2 Ep 190] θ̇ mean(abs) (last 10 eps): 3.8211, θ̇ max(abs): 8.0000, θ̇ min(abs): 0.0004, θ̇ signed_min: -8.0000, θ̇ signed_max: 8.0000
[Ep 240] NavR: -756.16, SafeR: -7.65, LossNav(min,max): (-17384.15,-17045.48), Violations: 17


Episode: 240, Eval Avg Nav. Reward: -1053.78, Eval Avg Safe. Reward: -17.05


[Stage2 Ep 200] θ̇ mean(abs) (last 10 eps): 1.8864, θ̇ max(abs): 8.0000, θ̇ min(abs): 0.0011, θ̇ signed_min: -8.0000, θ̇ signed_max: 8.0000
[Ep 250] NavR: -1708.31, SafeR: -3.04, LossNav(min,max): (-15250.79,-14947.91), Violations: 7


Episode: 250, Eval Avg Nav. Reward: -1108.95, Eval Avg Safe. Reward: -20.47
>>> 安定化制御込みの係数を保存しました (actor_safe_stable.pth) <<<
Game Done !! Max Eval Nav: -409.23, Max Eval Safe: -0.41
[SAVE] Saved run data to runs_cac/run_1.npz
===== RUN 1 END =====



In [20]:
!apt-get install -y ffmpeg

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [21]:
import torch

args = {
    'gym_name' : 'Pendulum-v1',
    'gamma': 0.99,
    'tau': 0.005,
    'alpha': 0.2,
    'seed': 123456,
    'batch_size': 256,
    'hidden_size': 256,
    'start_steps': 1000,
    'updates_per_step': 1,
    'target_update_interval': 1,
    'memory_size': 100000,
    'epochs': 100,
    'eval_interval': 10,
    'stage1_episodes': 50,
    'stage2_episodes': 200,
    'log_interval':  10,
}

env = gym.make(gym_name)
env.action_space.seed(seed)


agent = SoftActorCriticModel(state_num=env.observation_space.shape[0], action_num=env.action_space.shape[0],action_scale=env.action_space.high[0], args=args, device=device)
memory = ReplayMemory(args['memory_size'])

agent.actor_net.load_state_dict(torch.load("actor_safe_stable.pth"))
agent.actor_net.eval()
print(">>> actor_safe_only.pth をロードしました。安全制御のみのアクターでシミュレーション開始 <<<")


import numpy as np
import matplotlib.pyplot as plt
import math
import os

# --- シミュレーション（既存コードを想定） ---
env = gym.make(gym_name, render_mode='rgb_array')
env.action_space.seed(seed)

frames = []
# Gym のバージョン差に対応して reset の戻り値を扱う
reset_ret = env.reset()
if isinstance(reset_ret, tuple) and len(reset_ret) >= 1:
    state = reset_ret[0]
else:
    state = reset_ret

obs_list = []
done = False

for step in range(200):
    # action の取得（agent の実装に依存）
    action = agent.select_action(state, evaluate=True)
    step_ret = env.step(action)
    # Gym のバージョン差に対応して unpack
    if len(step_ret) == 5:
        next_state, reward, terminated, truncated, info = step_ret
    else:
        # 古いバージョン: (next_state, reward, done, info)
        next_state, reward, done_flag, info = step_ret
        terminated = done_flag
        truncated = False

    if terminated or truncated:
        done = True

    obs_list.append(next_state)

    # render は render_mode='rgb_array' のとき配列を返す
    try:
        frame = env.render()
        frames.append(frame)
    except Exception:
        # render が使えない環境でも続行
        pass

    state = next_state
    if done:
        print(f"Episode ended at step={step}")
        break

env.close()

print("Collected observations:", len(obs_list))
if len(obs_list) > 0:
    print("Sample obs[0]:", np.asarray(obs_list[0]).shape, np.asarray(obs_list[0]))

# --- 観測から cos, sin, theta_dot を抽出（堅牢処理） ---
cos_list = []
sin_list = []
thetadot_list = []

for obs in obs_list:
    obs = np.asarray(obs, dtype=float).ravel()
    # デフォルト値
    cos_theta = 0.0; sin_theta = 0.0; theta_dot = 0.0

    if obs.size >= 3:
        cos_theta = obs[0]
        sin_theta = obs[1]
        theta_dot = obs[2]
    elif obs.size == 2:
        theta = obs[0]
        theta_dot = obs[1]
        cos_theta = math.cos(theta)
        sin_theta = math.sin(theta)
    elif obs.size == 1:
        theta = obs[0]
        cos_theta = math.cos(theta)
        sin_theta = math.sin(theta)
        theta_dot = 0.0

    # 正規化（数値誤差対策）
    norm_cs = math.hypot(cos_theta, sin_theta)
    if norm_cs > 1e-6:
        cos_theta /= norm_cs
        sin_theta /= norm_cs

    cos_list.append(float(cos_theta))
    sin_list.append(float(sin_theta))
    thetadot_list.append(float(theta_dot))

# --- プロット ---
steps = np.arange(len(cos_list))
L = len(steps)
x_lst = [0, L]
y_lst = [-2.0, -2.0]
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
ax0, ax1, ax2 = axes

ax0.plot(steps, thetadot_list, color='tab:blue')
ax0.plot(x_lst, y_lst, linestyle='dashed', color='red', label='Safety threshold −2.0')
ax0.set_ylabel('Angular velocity (θ̇)')
ax0.grid(True)
ax0.set_title('Simulation: step vs θ̇, sinθ, cosθ')

ax1.plot(steps, sin_list, color='tab:orange')
ax1.set_ylabel('sin θ')
ax1.grid(True)

ax2.plot(steps, cos_list, color='tab:green')
ax2.set_ylabel('cos θ')
ax2.set_xlabel('Step')
ax2.grid(True)

plt.tight_layout()

# 表示（Jupyter なら表示、スクリプトならファイル保存して確認）
try:
    plt.show()
except Exception:
    pass

# 画像保存（必ず保存しておく）
out_dir = "./figs"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "sim_traces.png")
fig.savefig(out_path, dpi=200, bbox_inches='tight')
print("Saved plot to:", out_path)


>>> actor_safe_only.pth をロードしました。安全制御のみのアクターでシミュレーション開始 <<<


Episode ended at step=199
Collected observations: 200
Sample obs[0]: (3,) [0.1156367  0.99329156 1.3353219 ]


C:\Users\st_1o\AppData\Local\Temp\ipykernel_7968\1485422255.py:153: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved plot to: ./figs\sim_traces.png


In [22]:
import numpy as np
import matplotlib.pyplot as plt
import math
import os

env = gym.make(gym_name, render_mode='rgb_array')  # render_modeを指定。シミュレーション結果を画像の配列形式で返します（例えば、framesとしてフレームを保存できます）。
#env = gym.make(gym_name)
env.action_space.seed(seed)

frames = []
state, _ = env.reset()
done = False
obs_list = []  # ここに観測を貯める
for step in range(200):
#while not done:
    if step<0:
      action = env.action_space.sample()
    else:
      action = agent.select_action(state, evaluate=True)
    next_state, reward, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
      done=True

    # 観測を保存（state は step 前の観測、next_state は step 後）
    # ここでは next_state を保存する（任意で state を使っても良い）
    obs_list.append(next_state)

    #print(f"state={next_state}, action={action}")
    if done:
      print(f"break, step={step}")
     # break
    frames.append(env.render())
    state=next_state
env.close()
# --- 観測から cos, sin, theta_dot を抽出（堅牢処理） ---
cos_list = []
sin_list = []
thetadot_list = []

for obs in obs_list:
    obs = np.asarray(obs, dtype=float).ravel()
    if obs.size >= 3:
        # 典型的な Pendulum 形式を想定
        cos_theta = obs[0]
        sin_theta = obs[1]
        theta_dot = obs[2]
    elif obs.size == 2:
        # 角度のみと角速度が入っている等の別形式を想定（例: [theta, theta_dot]）
        theta = obs[0]
        theta_dot = obs[1]
        cos_theta = math.cos(theta)
        sin_theta = math.sin(theta)
    elif obs.size == 1:
        # 角度のみ
        theta = obs[0]
        cos_theta = math.cos(theta)
        sin_theta = math.sin(theta)
        theta_dot = 0.0
    else:
        # 予期しない形式はゼロ埋め
        cos_theta = 0.0; sin_theta = 0.0; theta_dot = 0.0

    # 正規化（数値誤差対策）：cos^2+sin^2 が 1 から大きく外れる場合は正規化
    norm_cs = math.hypot(cos_theta, sin_theta)
    if norm_cs > 1e-6:
        cos_theta /= norm_cs
        sin_theta /= norm_cs

    cos_list.append(cos_theta)
    sin_list.append(sin_theta)
    thetadot_list.append(theta_dot if 'theta_dot' in locals() else (obs[2] if obs.size>=3 else 0.0))

# --- プロット ---
steps = np.arange(len(cos_list))

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
ax0, ax1, ax2 = axes

ax0.plot(steps, thetadot_list, color='tab:blue')
ax0.set_ylabel('Angular velocity (θ̇)')
ax0.grid(True)
ax0.set_title('Simulation: step vs θ̇, sinθ, cosθ')

ax1.plot(steps, sin_list, color='tab:orange')
ax1.set_ylabel('sin θ')
ax1.grid(True)

ax2.plot(steps, cos_list, color='tab:green')
ax2.set_ylabel('cos θ')
ax2.set_xlabel('Step')
ax2.grid(True)

plt.tight_layout()
plt.show()

break, step=199


C:\Users\st_1o\AppData\Local\Temp\ipykernel_7968\562488994.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
from matplotlib import pyplot as plt
from matplotlib import animation

# 動画作成のセットアップ
fig = plt.figure()
patch = plt.imshow(frames[0], animated=True)

def update(frame):
    patch.set_data(frame)
    return patch,

ani = animation.FuncAnimation(

    fig, update, frames=frames, interval=100, blit=True
)

# 保存（mp4形式）
ani.save('pendulum_simulation.mp4', writer='ffmpeg')

MovieWriter ffmpeg unavailable; using Pillow instead.


ValueError: unknown file extension: .mp4

In [24]:
from IPython.display import Video

# 動画の再生
Video('pendulum_simulation.mp4', embed=True)

In [25]:
"""plt.figure(figsize=(10,5))
plt.plot(eps_s2, episode_dot_neg_counts, label="W1·W2 < 0 count", color='tab:purple')
plt.title("Negative dot product counts per episode")
plt.xlabel("Episode"); plt.ylabel("Count")
plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()"""

"""plt.figure(figsize=(10,5))
plt.plot(eps, episode_loss_safe, label="Safe Loss", color='tab:red')
plt.plot(eps, episode_loss_stability, label="Stability Loss", color='tab:blue')
plt.title("Safe vs Stability Loss over episodes")
plt.xlabel("Episode"); plt.ylabel("Loss value")
plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()"""

'plt.figure(figsize=(10,5))\nplt.plot(eps, episode_loss_safe, label="Safe Loss", color=\'tab:red\')\nplt.plot(eps, episode_loss_stability, label="Stability Loss", color=\'tab:blue\')\nplt.title("Safe vs Stability Loss over episodes")\nplt.xlabel("Episode"); plt.ylabel("Loss value")\nplt.legend(); plt.grid(True)\nplt.tight_layout(); plt.show()'

In [26]:
'''#-- 全エピソード終了後に可視化 ---
eps = range(1, total_episodes+1)
plt.figure(figsize=(12, 8))

plt.subplot(2,2,1)
plt.plot(eps, episode_rewards_nav, label="Nav Reward")
plt.plot(eps, episode_rewards_safe, label="Safe Reward")
plt.legend(); plt.title("Episode Rewards")

plt.subplot(2,2,2)
plt.plot(eps, episode_loss_min, label="LossNav Min")
plt.plot(eps, episode_loss_max, label="LossNav Max")
plt.legend(); plt.title("LossNav Range")

plt.subplot(2,2,3)
plt.plot(eps, episode_violations, label="Violations")
plt.legend(); plt.title("Barrier Violations")

plt.tight_layout()
plt.show()

# --- 最終可視化に最小値・符号付き min/max を追加するブロック（既存の theta-dot visualization の拡張） ---
# [ADDED START] Theta-dot visualization for Stage2 (extended)
if len(episode_theta_dot_mean) > 0:
    eps_stage2 = range(args['stage1_episodes'] + 1, args['stage1_episodes'] + 1 + len(episode_theta_dot_mean))
    plt.figure(figsize=(10,5))
    plt.plot(eps_stage2, episode_theta_dot_mean, label="θ̇ mean (abs)", color='tab:orange')
    plt.plot(eps_stage2, episode_theta_dot_max,  label="θ̇ max (abs)", color='tab:red')
    plt.plot(eps_stage2, episode_theta_dot_min,  label="θ̇ min (abs)", color='tab:purple')    # [ADDED]
    plt.axvline(x=args['stage1_episodes'], color='gray', linestyle='--', label="Stage1/Stage2 split")
    plt.xlabel("Episode")
    plt.ylabel("Angular velocity (abs)")
    plt.title("Stage2 Angular Velocity per Episode (abs stats)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Signed min/max plot (directional)
    plt.figure(figsize=(10,3))
    plt.plot(eps_stage2, episode_theta_dot_signed_min, label="θ̇ signed min", color='tab:blue')
    plt.plot(eps_stage2, episode_theta_dot_signed_max, label="θ̇ signed max", color='tab:green')
    plt.axvline(x=args['stage1_episodes'], color='gray', linestyle='--')
    plt.xlabel("Episode")
    plt.ylabel("Angular velocity (signed)")
    plt.title("Stage2 Angular Velocity per Episode (signed min/max)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No Stage2 theta-dot data collected.")
# [ADDED END]
import numpy as np
import matplotlib.pyplot as plt

def safe_series_all_episodes():
    # combine Stage1 safe grad (per-episode) and Stage2 safe grad (per-episode)
    safe_all = []
    # stage1 entries correspond to episodes 1..stage1_episodes
    safe_all.extend(episode_safe_grad_stage1)
    # stage2 entries correspond to episodes stage1+1 ...
    safe_all.extend(episode_safe_grad_stage2)
    # pad/truncate to total_episodes length if necessary
    if len(safe_all) < total_episodes:
        safe_all.extend([0.0] * (total_episodes - len(safe_all)))
    return np.array(safe_all[:total_episodes])

# prepare series
eps = np.arange(1, total_episodes + 1)
safe_all = safe_series_all_episodes()

# stability only for Stage2 episodes
eps_s2 = np.arange(args['stage1_episodes'] + 1, args['stage1_episodes'] + 1 + len(episode_stability_grad))
stability_s2 = np.array(episode_stability_grad)

# dot product series already collected per Stage2 episode (signed)
dot_s2 = np.array(episode_grad_dot)

# compute cosine similarity per Stage2 episode robustly
# Use stored norms: episode_safe_grad_stage2 (|W1|) and episode_stability_grad (|W2|)
w1 = np.array(episode_safe_grad_stage2) if len(episode_safe_grad_stage2) > 0 else np.zeros_like(dot_s2)
w2 = np.array(episode_stability_grad) if len(episode_stability_grad) > 0 else np.zeros_like(dot_s2)
den = (w1 * w2) + 1e-12
cos_s2 = np.divide(dot_s2, den, out=np.zeros_like(dot_s2), where=den!=0)

# compute |W1| * cosθ (per Stage2 episode)
w1_cos_s2 = w1 * cos_s2

# --- Print quick stats for debugging ---
def print_grad_stats(name, arr):
    a = np.array(arr) if len(arr) > 0 else np.array([0.0])
    print(f"{name}: mean={a.mean():.6f}, median={np.median(a):.6f}, std={a.std():.6f}, min={a.min():.6f}, max={a.max():.6f}")

print("=== Gradient statistics (Stage2) ===")
print_grad_stats("Safety grad (Stage2)", episode_safe_grad_stage2)
print_grad_stats("Stability grad (Stage2)", episode_stability_grad)
print_grad_stats("Dot product (Stage2)", episode_grad_dot)
#print_grad_stats("Cosine similarity (Stage2)", cos_s2.tolist())

# --- Plots ---
plt.figure(figsize=(14, 10))

# 1: Safe grad across all episodes
plt.subplot(2,2,1)
plt.plot(eps, safe_all, label="Safe grad norm (all episodes)", color='tab:blue')
plt.axvline(x=args['stage1_episodes'], color='gray', linestyle='--', label="Stage1/Stage2 split")
plt.title("Safe gradient norm across all episodes")
plt.xlabel("Episode"); plt.ylabel("L2 Norm")
plt.legend(); plt.grid(True)

# 2: Stability grad only (Stage2)
"""plt.subplot(2,2,2)
if len(eps_s2) > 0:
    plt.plot(eps_s2, stability_s2, label="Stability grad norm (Stage2)", color='tab:green')
    plt.plot(eps_s2, np.log10(stability_s2 + 1e-12), label="log10 |W2|", color='tab:red', linestyle='--')
plt.title("Stability gradient norm (Stage2 only)")
plt.xlabel("Episode"); plt.ylabel("L2 Norm / log10 Norm")
plt.legend(); plt.grid(True)"""
fig, ax1 = plt.subplots(figsize=(10,5))

# 左軸：生ノルム (線)
ax1.plot(eps_s2, episode_safe_grad_stage2, color='tab:blue', label='W2 norm (raw)')
ax1.set_xlabel('Episode')
ax1.set_ylabel('W2 norm (L2)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# 右軸：log10 (破線)
ax2 = ax1.twinx()
ax2.plot(eps_s2, episode_stability_grad_log10, color='tab:orange', linestyle='--', label='log10(W2 norm)')
ax2.set_ylabel('log10(W2 norm)', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

# 凡例（両軸の線をまとめる）
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')

plt.title("W2 (Stability) norm and log10(W2) (Stage2)")
plt.grid(True)
plt.tight_layout()
plt.show()


# 3: Dot product (Stage2)
plt.subplot(2,2,3)
if len(eps_s2) > 0:
    plt.plot(eps_s2, dot_s2, label="dot(grad_st, grad_sa) (signed)", color='tab:purple')
    plt.plot(eps_s2, np.abs(dot_s2), label="|dot| (abs)", color='tab:orange')
plt.title("Dot product between stability and safety gradients (Stage2)")
plt.xlabel("Episode"); plt.ylabel("Dot value")
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

# --- Optional: separate long-run safe vs stability per-episode history (existing history lists) ---
if hasattr(agent, 'history_safe_grad_norms') and len(agent.history_safe_grad_norms) > 0:
    plt.figure(figsize=(10,4))
    eps_hist = np.arange(1, len(agent.history_safe_grad_norms) + 1)
    plt.plot(eps_hist, agent.history_safe_grad_norms, label="Safe grad norm (per episode)", color='tab:blue')
    plt.plot(eps_hist, agent.history_stability_grad_norms, label="Stability grad norm (per episode)", color='tab:green')
    plt.axvline(x=args['stage1_episodes'], color='red', linestyle='--', label="Stage transition")
    plt.title("Safe vs Stability gradient norms by Episode (history)")
    plt.xlabel("Episode"); plt.ylabel("L2 Norm")
    plt.legend(); plt.grid(True)
    plt.tight_layout()
    plt.show()

# --- Run-level bundle save block ---

# base output directory (任意に変更)
base_out_dir = "experiment_runs"
os.makedirs(base_out_dir, exist_ok=True)

# run-specific folder name (一意化: gym名, 総エピソード数, タイムスタンプ, optional tag)
ts = time.strftime("%Y%m%d-%H%M%S")
tag = f"{gym_name if 'gym_name' in globals() else 'run'}_ep{total_episodes}_{ts}"
run_dir = os.path.join(base_out_dir, tag)
os.makedirs(run_dir, exist_ok=True)

def _safe_array(x):
    try:
        return np.array(x)
    except Exception:
        return np.array([])

def save_figures_to_folder(folder, prefix="fig"):
    saved = []
    for i, fig_num in enumerate(plt.get_fignums(), start=1):
        try:
            fig = plt.figure(fig_num)
            fname_base = f"{prefix}{i}"
            png_path = os.path.join(folder, fname_base + ".png")
            pdf_path = os.path.join(folder, fname_base + ".pdf")
            fig.savefig(png_path, dpi=200, bbox_inches='tight')
            fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
            saved.append((png_path, pdf_path))
        except Exception as e:
            print(f"[SAVE ERROR] figure {fig_num}: {e}")
        finally:
            try:
                plt.close(fig)
            except Exception:
                pass
    return saved

def save_data_bundle(folder, prefix="data"):
    data_dict = {
        "episode_rewards_nav": _safe_array(globals().get("episode_rewards_nav", [])),
        "episode_rewards_safe": _safe_array(globals().get("episode_rewards_safe", [])),
        "episode_loss_min": _safe_array(globals().get("episode_loss_min", [])),
        "episode_loss_max": _safe_array(globals().get("episode_loss_max", [])),
        "episode_violations": _safe_array(globals().get("episode_violations", [])),
        "episode_safe_grad_stage1": _safe_array(globals().get("episode_safe_grad_stage1", [])),
        "episode_safe_grad_stage2": _safe_array(globals().get("episode_safe_grad_stage2", [])),
        "episode_stability_grad": _safe_array(globals().get("episode_stability_grad", [])),
        "episode_grad_dot": _safe_array(globals().get("episode_grad_dot", [])),
        "episode_grad_cos": _safe_array(globals().get("episode_grad_cos", [])),
        "episode_theta_dot_mean": _safe_array(globals().get("episode_theta_dot_mean", [])),
        "episode_theta_dot_max": _safe_array(globals().get("episode_theta_dot_max", [])),
        "episode_theta_dot_min": _safe_array(globals().get("episode_theta_dot_min", [])),
        "episode_theta_dot_signed_min": _safe_array(globals().get("episode_theta_dot_signed_min", [])),
        "episode_theta_dot_signed_max": _safe_array(globals().get("episode_theta_dot_signed_max", [])),
        "episode_loss_safe": _safe_array(globals().get("episode_loss_safe", [])),
        "episode_loss_stability": _safe_array(globals().get("episode_loss_stability", [])),
        "episode_dot_neg_counts": _safe_array(globals().get("episode_dot_neg_counts", [])),
        "episode_dot_neg_ratio": _safe_array(globals().get("episode_dot_neg_ratio", [])),
        "episode_safe_grad_log10": _safe_array(globals().get("episode_safe_grad_log10", [])),
    }

    npz_path = os.path.join(folder, f"{prefix}.npz")
    try:
        np.savez_compressed(npz_path, **data_dict)
    except Exception as e:
        print(f"[SAVE ERROR] npz: {e}")

    csv_path = os.path.join(folder, f"{prefix}.csv")
    header = list(data_dict.keys())
    max_len = max((len(v) for v in data_dict.values()), default=0)
    try:
        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["episode_index"] + header)
            for i in range(max_len):
                row = [i + 1]
                for key in header:
                    arr = data_dict[key]
                    row.append(float(arr[i]) if i < len(arr) else "")
                writer.writerow(row)
    except Exception as e:
        print(f"[SAVE ERROR] csv: {e}")

    # also save a small metadata file (json-like plain text)
    meta_path = os.path.join(folder, "meta.txt")
    try:
        with open(meta_path, "w") as mf:
            mf.write(f"tag: {tag}\n")
            mf.write(f"total_episodes: {total_episodes}\n")
            if 'args' in globals():
                mf.write("args:\n")
                for k, v in args.items():
                    mf.write(f"  {k}: {v}\n")
    except Exception as e:
        print(f"[SAVE ERROR] meta: {e}")

    return npz_path, csv_path, meta_path

# perform the run-bundle save (呼び出し)
saved_figs = save_figures_to_folder(run_dir, prefix="fig_")
npz_path, csv_path, meta_path = save_data_bundle(run_dir, prefix="training_data")

# report
if saved_figs:
    for png_path, pdf_path in saved_figs:
        print(f"Saved PNG: {os.path.abspath(png_path)}")
        print(f"Saved PDF: {os.path.abspath(pdf_path)}")
else:
    print("No matplotlib figures to save (plt.get_fignums() returned empty).")

print(f"Saved NPZ: {os.path.abspath(npz_path)}")
print(f"Saved CSV: {os.path.abspath(csv_path)}")
print(f"Saved META: {os.path.abspath(meta_path)}")
print(f"Run bundle saved to folder: {os.path.abspath(run_dir)}")
# --- End run-level bundle save block ---
plt.figure(figsize=(10,5))
plt.plot(eps, episode_loss_safe, label="Safe Loss", color='tab:red')
plt.title("Safe Loss over episodes")
plt.xlabel("Episode"); plt.ylabel("Loss value")
plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

plt.figure(figsize=(10,5))
plt.plot(eps, episode_loss_stability, label="Stability Loss", color='tab:blue')
plt.title("Stability Loss over episodes")
plt.xlabel("Episode"); plt.ylabel("Loss value")
plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

plt.figure(figsize=(10,5))
plt.plot(eps_s2, episode_dot_neg_ratio, label="W1·W2 < 0 ratio", color='tab:orange')
plt.title("Negative dot product ratio per episode")
plt.xlabel("Episode"); plt.ylabel("Ratio (0.0–1.0)")
plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

# --- 2x2 subplot にまとめて描画して保存するコード（差し替え用） ---
import numpy as np
import os
import matplotlib.pyplot as plt

def _ensure_np(x):
    return np.array(x) if x is not None else np.array([])

def _finite_to_nan(a):
    a = np.array(a, dtype=float)
    a[~np.isfinite(a)] = np.nan
    return a

def _trim_min_len(x, *arrays):
    minlen = min([len(x)] + [len(a) for a in arrays])
    return x[:minlen], [a[:minlen] for a in arrays]

# --- Prepare sequences ---
eps = np.arange(1, total_episodes + 1)
safe_all = safe_series_all_episodes() if 'safe_series_all_episodes' in globals() else _ensure_np(episode_safe_grad_stage1).tolist() + _ensure_np(episode_safe_grad_stage2).tolist()
safe_all = _ensure_np(safe_all)

eps_s2 = np.arange(args['stage1_episodes'] + 1, args['stage1_episodes'] + 1 + len(episode_stability_grad))
w2_raw = _ensure_np(episode_stability_grad)
w2_log10 = _ensure_np(episode_stability_grad_log10)
w1_stage2 = _ensure_np(episode_safe_grad_stage2)
dot_signed = _ensure_np(episode_grad_dot)

# sanitize non-finite -> NaN
safe_all = _finite_to_nan(safe_all)
w2_raw = _finite_to_nan(w2_raw)
w2_log10 = _finite_to_nan(w2_log10)
w1_stage2 = _finite_to_nan(w1_stage2)
dot_signed = _finite_to_nan(dot_signed)

# Align Stage2 arrays to same min length (eps_s2 based)
eps_s2, [w2_raw, w2_log10, w1_stage2, dot_signed] = _trim_min_len(eps_s2, w2_raw, w2_log10, w1_stage2, dot_signed)

# --- Create 2x2 figure ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax1 = axes[0,0]; ax2 = axes[0,1]; ax3 = axes[1,0]; ax4 = axes[1,1]

# (1) Safe grad across all episodes
ax1.plot(eps, safe_all, label="Safe grad norm (all episodes)", color='tab:blue')
ax1.axvline(x=args['stage1_episodes'], color='gray', linestyle='--', label="Stage1/Stage2 split")
ax1.set_title("Safe gradient norm across all episodes")
ax1.set_xlabel("Episode"); ax1.set_ylabel("L2 Norm")
ax1.legend(); ax1.grid(True)

# (2) W2 raw and log10 on twin axis (Stage2)
ax2.plot(eps_s2, w2_raw, color='tab:blue', label='W2 norm (raw)')
ax2.set_xlabel("Episode")
ax2.set_ylabel("W2 norm (L2)", color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')
ax2.grid(True, which='both', linestyle=':', alpha=0.3)

ax2r = ax2.twinx()
ax2r.plot(eps_s2, w2_log10, color='tab:orange', linestyle='--', label='log10(W2 norm)')
ax2r.set_ylabel('log10(W2 norm)', color='tab:orange')
ax2r.tick_params(axis='y', labelcolor='tab:orange')

# combined legend for ax2
lines_l, labels_l = ax2.get_legend_handles_labels()
lines_r, labels_r = ax2r.get_legend_handles_labels()
if lines_l or lines_r:
    ax2.legend(lines_l + lines_r, labels_l + labels_r, loc='upper right')
ax2.set_title("W2 (Stability) norm and log10(W2) (Stage2)")

# (3) Dot product signed and abs (Stage2)
ax3.plot(eps_s2, dot_signed, label='dot(grad_st, grad_sa) (signed)', color='tab:purple')
ax3.plot(eps_s2, np.abs(dot_signed), label='|dot| (abs)', color='tab:orange', alpha=0.9)
ax3.axhline(0.0, color='k', linestyle='--', linewidth=0.6)
ax3.set_title("Dot product between stability and safety gradients (Stage2)")
ax3.set_xlabel("Episode"); ax3.set_ylabel("Dot value")
ax3.legend(); ax3.grid(True)

# (4) Theta-dot summary or placeholder
if len(episode_theta_dot_mean) > 0:
    eps_stage2 = np.arange(args['stage1_episodes'] + 1, args['stage1_episodes'] + 1 + len(episode_theta_dot_mean))
    ax4.plot(eps_stage2, _finite_to_nan(episode_theta_dot_mean), label="θ̇ mean (abs)", color='tab:orange')
    ax4.plot(eps_stage2, _finite_to_nan(episode_theta_dot_max),  label="θ̇ max (abs)", color='tab:red')
    ax4.plot(eps_stage2, _finite_to_nan(episode_theta_dot_min),  label="θ̇ min (abs)", color='tab:purple')
    ax4.set_title("Stage2 Angular Velocity per Episode (abs stats)")
    ax4.set_xlabel("Episode"); ax4.set_ylabel("Angular velocity (abs)")
    ax4.legend(); ax4.grid(True)
else:
    ax4.text(0.5, 0.5, "No Stage2 θ̇ data", ha='center', va='center', fontsize=12, color='gray')
    ax4.set_xticks([]); ax4.set_yticks([])
    ax4.set_title("Stage2 Angular Velocity (no data)")

plt.tight_layout()

# --- Save single combined figure as PNG + PDF ---
os.makedirs(run_dir, exist_ok=True)
combined_name = f"combined_plots_{ts}"
png_path = os.path.join(run_dir, combined_name + ".png")
pdf_path = os.path.join(run_dir, combined_name + ".pdf")
try:
    fig.savefig(png_path, dpi=200, bbox_inches='tight')
    fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
    print(f"Saved combined PNG: {os.path.abspath(png_path)}")
    print(f"Saved combined PDF: {os.path.abspath(pdf_path)}")
except Exception as e:
    print(f"[FIG SAVE ERROR] {e}")
finally:
    plt.close(fig)

# --- Save data bundle (npz/csv/meta) using your function (updates expected) ---
try:
    npz_path, csv_path, meta_path = save_data_bundle(run_dir, prefix="training_data")
    if npz_path: print(f"Saved NPZ: {os.path.abspath(npz_path)}")
    if csv_path: print(f"Saved CSV: {os.path.abspath(csv_path)}")
    if meta_path: print(f"Saved META: {os.path.abspath(meta_path)}")
except Exception as e:
    print(f"[SAVE BUNDLE ERROR] {e}")'''

'#-- 全エピソード終了後に可視化 ---\neps = range(1, total_episodes+1)\nplt.figure(figsize=(12, 8))\n\nplt.subplot(2,2,1)\nplt.plot(eps, episode_rewards_nav, label="Nav Reward")\nplt.plot(eps, episode_rewards_safe, label="Safe Reward")\nplt.legend(); plt.title("Episode Rewards")\n\nplt.subplot(2,2,2)\nplt.plot(eps, episode_loss_min, label="LossNav Min")\nplt.plot(eps, episode_loss_max, label="LossNav Max")\nplt.legend(); plt.title("LossNav Range")\n\nplt.subplot(2,2,3)\nplt.plot(eps, episode_violations, label="Violations")\nplt.legend(); plt.title("Barrier Violations")\n\nplt.tight_layout()\nplt.show()\n\n# --- 最終可視化に最小値・符号付き min/max を追加するブロック（既存の theta-dot visualization の拡張） ---\n# [ADDED START] Theta-dot visualization for Stage2 (extended)\nif len(episode_theta_dot_mean) > 0:\n    eps_stage2 = range(args[\'stage1_episodes\'] + 1, args[\'stage1_episodes\'] + 1 + len(episode_theta_dot_mean))\n    plt.figure(figsize=(10,5))\n    plt.plot(eps_stage2, episode_theta_dot_mean, label="θ̇ mean (abs)", colo